In [2]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [3]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
306,"I watched Love Life on holiday, when it was fi...",positive
361,Deeply humorous yet honest comedy about a bunc...,positive
629,Average (and surprisingly tame) Fulci giallo w...,negative
245,This movie has got to be one of the worst I ha...,negative
653,"One of the funniest, most romantic, and most m...",positive


In [7]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [9]:
df = normalize_text(df)
df.head()

,review,sentiment
306,watched love life holiday filmed film festival...,positive
361,deeply humorous yet honest comedy bunch grownu...,positive
629,average surprisingly tame fulci giallo mean st...,negative
245,movie got one worst ever seen make dvd story l...,negative
653,one funniest romantic musical movie ever defin...,positive


In [10]:
df['sentiment'].value_counts()


sentiment
negative    257
positive    243
Name: count, dtype: int64

In [11]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [12]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
306,watched love life holiday filmed film festival...,1
361,deeply humorous yet honest comedy bunch grownu...,1
629,average surprisingly tame fulci giallo mean st...,0
245,movie got one worst ever seen make dvd story l...,0
653,one funniest romantic musical movie ever defin...,1


In [13]:
df.isnull().sum()


review       0
sentiment    0
dtype: int64

In [14]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [18]:
import dagshub
import mlflow

mlflow.set_tracking_uri("https://dagshub.com/osamashabih6960/Youtube-Capstone-Project-using-Mlops-.mlflow")

dagshub.init(
    repo_owner="osamashabih6960",
    repo_name="Youtube-Capstone-Project-using-Mlops-",
    mlflow=True
)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")

Initialized MLflow to track repo "osamashabih6960/Youtube-Capstone-Project-using-Mlops-"

Repository osamashabih6960/Youtube-Capstone-Project-using-Mlops- initialized!

2026/03/11 16:35:26 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/5eaac4c1bfb84dd68e70d3459fca3bcf', creation_time=1773227127369, experiment_id='0', last_update_time=1773227127369, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, workspace='default'>

In [19]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)
    

2026-03-11 16:36:23,164 - INFO - Starting MLflow run...
2026-03-11 16:36:24,178 - INFO - Logging preprocessing parameters...
2026-03-11 16:36:25,685 - INFO - Initializing Logistic Regression model...
2026-03-11 16:36:25,687 - INFO - Fitting the model...
2026-03-11 16:36:25,757 - INFO - Model training complete.
2026-03-11 16:36:25,758 - INFO - Logging model parameters...
2026-03-11 16:36:26,235 - INFO - Making predictions...
2026-03-11 16:36:26,237 - INFO - Calculating evaluation metrics...
2026-03-11 16:36:26,253 - INFO - Logging evaluation metrics...
2026-03-11 16:36:28,225 - INFO - Saving and logging the model...
2026/03/11 16:36:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/11 16:36:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

🏃 View run victorious-mole-960 at: https://dagshub.com/osamashabih6960/Youtube-Capstone-Project-using-Mlops-.mlflow/#/experiments/0/runs/ab3ffe77a94d4202a37e4ba8e1dd5557
🧪 View experiment at: https://dagshub.com/osamashabih6960/Youtube-Capstone-Project-using-Mlops-.mlflow/#/experiments/0
